# Demostración del Pipeline RAG: Entity Resolution en Banca de Inversión
Este cuaderno inicializa la base de datos vectorial (FAISS + BM25) y ejecuta una serie de *Edge Cases* (casos extremos) para demostrar la robustez de las reglas de negocio estrictas y el cálculo del *confidence score*.

In [1]:
import sys
import os
from pathlib import Path

# Configuramos el path dinámicamente para que reconozca la carpeta 'src' desde la subcarpeta 'notebooks'
# Esto garantiza que el evaluador pueda ejecutar el notebook sin errores de "ModuleNotFoundError"
project_root = Path(os.getcwd()).parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print(f"Raíz del proyecto configurada en: {project_root}")

Raíz del proyecto configurada en: /home/ema/Documentos/master/genAI_project


In [ ]:
import json
import time
from src.indexer import DBIndexer
from src.rag_pipeline import EntityResolutionPipeline
from src.enrichment_agent import EnrichmentAgent


print("=== FASE 1: INICIALIZACIÓN DE LA BASE DE DATOS ===")
indexer = DBIndexer()

try:
    indexer.load_indices()
    print("✅ Índices cargados exitosamente desde almacenamiento local (vectorstore/).")
except FileNotFoundError:
    print("⚠️ Índices no encontrados. Construyendo FAISS y BM25 desde cero...")
    indexer.build_and_save_indices()

/home/ema/Documentos/master/genAI_project/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


=== FASE 1: INICIALIZACIÓN DE LA BASE DE DATOS ===


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7692.54it/s]


✅ Índices cargados exitosamente desde almacenamiento local (vectorstore/).


In [3]:
print("=== FASE 2: EVALUACIÓN DE CASOS EXTREMOS (EDGE CASES) ===")
pipeline = EntityResolutionPipeline()

test_cases = [
    {
        "id": "Test 1: Match Exacto con Ruido Sintáctico",
        "query": "  Al Sariya Commercial Investments L.L.C. ",
        "desc": "El preprocesador limpia los espacios y normaliza L.L.C. a LLC. LangChain debe hacer match perfecto."
    },
    {
        "id": "Test 2: Falso Positivo por Serie/Fondo Distinto",
        "query": "Border to Coast Bedfordshire Fund II LP",
        "desc": "Existe 'Border to Coast Bedfordshire LP' en la DB. Al añadir 'Fund II', es una entidad legal distinta. Debe rechazarlo y pedir revisión."
    },
    {
        "id": "Test 3: Falso Positivo por Diferencia Geográfica",
        "query": "Caisse Regionale Crédit Agricole Mutuel de Paris",
        "desc": "Existe '... Mutuel des Côtes d'Armor' en la DB. El cambio de ciudad implica subsidiaria distinta. Debe rechazarlo."
    },
    {
        "id": "Test 4: Entidad Totalmente Inexistente",
        "query": "Tech Ventures Capital Innovacion SA",
        "desc": "No existe nada semánticamente parecido en el CSV. El modelo debe asignar is_match = False."
    }
]

for case in test_cases:
    print(f"{'='*70}")
    print(f"🚀 EJECUTANDO {case['id']}")
    print(f"📥 Input LP (Capital Call): '{case['query']}'")
    print(f"{'-'*70}")

    start_time = time.time()
    result = pipeline.resolve_entity(case["query"])
    elapsed_time = time.time() - start_time

    print(f"⏱️ Tiempo de resolución: {elapsed_time:.2f}s")
    print("📊 SALIDA JSON:")
    print(json.dumps(result, indent=2, ensure_ascii=False))

    conf_score = result.get("confidence_score", 1.0)
    human_review = result.get("trigger_human_review")
    
    print(f"\n[Auditoría]: Confidence Score = {conf_score}")
    if conf_score < 0.85 and human_review is True:
        print("✅ REGLA CUMPLIDA: Se forzó correctamente la revisión humana.")
    elif conf_score >= 0.85 and human_review is False:
        print("✅ ALTA CONFIANZA: Resolución automática aprobada.")

=== FASE 2: EVALUACIÓN DE CASOS EXTREMOS (EDGE CASES) ===
🔌 Inicializando LLM: Groq (llama-3.3-70b-versatile)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7648.15it/s]


🚀 EJECUTANDO Test 1: Match Exacto con Ruido Sintáctico
📥 Input LP (Capital Call): '  Al Sariya Commercial Investments L.L.C. '
----------------------------------------------------------------------

🔍 Resolviendo Entidad Original: '  Al Sariya Commercial Investments L.L.C. ' (Query procesada: 'AL SARIYA COMMERCIAL INVESTMENTS LLC.')...


INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


⏱️ Tiempo de resolución: 0.49s
📊 SALIDA JSON:
{
  "resolved_name": "Al Sariya Commercial Investments LLC",
  "is_match": true,
  "confidence_score": 0.9993199,
  "trigger_human_review": false,
  "metadata_extracted": {
    "id": 0,
    "relevance_score": 0.9993199,
    "investor_type": "SWF",
    "country": "United Arab Emirates",
    "sp_rating": "AA",
    "moodys_rating": "NR"
  }
}

[Auditoría]: Confidence Score = 0.9993199
✅ ALTA CONFIANZA: Resolución automática aprobada.
🚀 EJECUTANDO Test 2: Falso Positivo por Serie/Fondo Distinto
📥 Input LP (Capital Call): 'Border to Coast Bedfordshire Fund II LP'
----------------------------------------------------------------------

🔍 Resolviendo Entidad Original: 'Border to Coast Bedfordshire Fund II LP' (Query procesada: 'BORDER TO COAST BEDFORDSHIRE FUND II LP')...


...

⏱️ Tiempo de resolución: 0.34s
📊 SALIDA JSON:
{
  "resolved_name": null,
  "is_match": false,
  "confidence_score": 0.8,
  "trigger_human_review": true,
  "metadata_extracted": {}
}

[Auditoría]: Confidence Score = 0.8
✅ REGLA CUMPLIDA: Se forzó correctamente la revisión humana.
🚀 EJECUTANDO Test 3: Falso Positivo por Diferencia Geográfica
📥 Input LP (Capital Call): 'Caisse Regionale Crédit Agricole Mutuel de Paris'
----------------------------------------------------------------------

🔍 Resolviendo Entidad Original: 'Caisse Regionale Crédit Agricole Mutuel de Paris' (Query procesada: 'CAISSE REGIONALE CREDIT AGRICOLE MUTUEL DE PARIS')...


INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


⏱️ Tiempo de resolución: 0.57s
📊 SALIDA JSON:
{
  "resolved_name": "Caisse Regionale Crédit Agricole Mutuel des Côtes d'Armor",
  "is_match": false,
  "confidence_score": 0.8,
  "trigger_human_review": true,
  "metadata_extracted": {}
}

[Auditoría]: Confidence Score = 0.8
✅ REGLA CUMPLIDA: Se forzó correctamente la revisión humana.
🚀 EJECUTANDO Test 4: Entidad Totalmente Inexistente
📥 Input LP (Capital Call): 'Tech Ventures Capital Innovacion SA'
----------------------------------------------------------------------

🔍 Resolviendo Entidad Original: 'Tech Ventures Capital Innovacion SA' (Query procesada: 'TECH VENTURES CAPITAL INNOVACION SA')...


INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


⏱️ Tiempo de resolución: 0.64s
📊 SALIDA JSON:
{
  "resolved_name": null,
  "is_match": false,
  "confidence_score": 0.0,
  "trigger_human_review": true,
  "metadata_extracted": {}
}

[Auditoría]: Confidence Score = 0.0
✅ REGLA CUMPLIDA: Se forzó correctamente la revisión humana.


In [ ]:
print("=== FASE 3: ENRIQUECIMIENTO WEB (ONBOARDING DE NUEVO LP) ===")

# Instanciamos nuestro nuevo Agente
agent = EnrichmentAgent()

# Caso de prueba: Un fondo real que no está en la base de datos interna
nuevo_lp_entrante = "Andreessen Horowitz"

print(f"{'='*70}")
print(f"🚀 INICIANDO PROTOCOLO DE ALTA PARA: '{nuevo_lp_entrante}'")
print(f"{'-'*70}")

# Ejecutamos el agente
enrichment_result = agent.run_enrichment(nuevo_lp_entrante)

print("\n📊 BORRADOR DE ALTA GENERADO (JSON):")
print(json.dumps(enrichment_result, indent=2, ensure_ascii=False))

if enrichment_result.get("enrichment_success"):
    print("\n✅ ÉXITO: El agente ha propuesto un alta válida lista para revisión humana.")
else:
    print("\n❌ RECHAZO: El agente determinó que la entidad no tiene huella financiera válida.")

=== FASE 3: ENRIQUECIMIENTO WEB (ONBOARDING DE NUEVO LP) ===
🔌 Inicializando LLM: Groq (llama-3.3-70b-versatile)
🚀 INICIANDO PROTOCOLO DE ALTA PARA: 'Andreessen Horowitz'
----------------------------------------------------------------------
🌐 [Tavily Search] Investigando la huella digital de: 'Andreessen Horowitz'...


/home/ema/Documentos/master/genAI_project/src/enrichment_agent.py:48: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  self.search_tool = TavilySearchResults(max_results=3)


🧠 [LLM] Analizando y estructurando resultados web...


INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"



📊 BORRADOR DE ALTA GENERADO (JSON):
{
  "proposed_official_name": "Andreessen Horowitz",
  "country": "United States",
  "investor_type": "Venture Capital",
  "description_summary": "Firma de capital de riesgo que invierte en empresas de tecnología desde la etapa de semilla hasta la etapa tardía.",
  "enrichment_success": true
}

✅ ÉXITO: El agente ha propuesto un alta válida lista para revisión humana.
